**Membuat SparkSession**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

spark = SparkSession.builder \
    .appName("Tugas4-AnalisisTransaksi") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/09/10 18:50:21 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/10 18:50:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 18:50:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Membaca dan Eksplorasi Awal**

In [3]:
# Membaca dataset langsung dari HDFS
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

**B. Menangani Data Kosong**

Kolom rating memiliki nilai kosong (NaN/null). Berikut jumlah baris yang kosong, lalu penanganannya.

In [4]:
# Menghitung jumlah baris dengan rating kosong
jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah baris dengan rating kosong:", jumlah_kosong)

Jumlah baris dengan rating kosong: 204


Pilihan: menggunakan df.na.fill() untuk mengisi rating yang kosong dengan nilai rata-rata rating yang tersedia (bukan df.na.drop()).

Alasan: baris dengan rating kosong tetap mengandung informasi penting pada kolom lain (total_pendapatan, kategori, kota, dst). Jika baris tersebut dibuang (na.drop()), seluruh transaksi ikut hilang dari analisis pendapatan dan tier, padahal ketiadaan rating kemungkinan besar hanya berarti pembeli tidak memberi ulasan, bukan bahwa transaksinya tidak valid. Mengisi (na.fill()) dengan rata-rata rating mempertahankan seluruh baris data sekaligus meminimalkan distorsi terhadap rata-rata keseluruhan.

In [5]:
# Menghitung rata-rata rating (mengabaikan nilai null secara otomatis)
rata_rata_rating = df.select(avg("rating")).first()[0]
rata_rata_rating = round(rata_rata_rating, 2)
print("Rata-rata rating (tanpa nilai kosong):", rata_rata_rating)

# Mengisi nilai kosong pada kolom rating dengan rata-rata
df = df.na.fill({"rating": rata_rata_rating})

# Verifikasi tidak ada lagi nilai kosong
print("Sisa baris dengan rating kosong:", df.filter(col("rating").isNull()).count())

Rata-rata rating (tanpa nilai kosong): 4.15
Sisa baris dengan rating kosong: 0


**C. Transformasi Data**

In [12]:
# Menambahkan kolom total_pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Menambahkan kolom tier_transaksi berdasarkan total_pendapatan
df = df.withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



**D. Analisis dengan GroupBy**

D1. Kategori dengan total_pendapatan tertinggi

In [13]:
pendapatan_per_kategori = df.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc())

pendapatan_per_kategori.show()
print("Kategori dengan total_pendapatan tertinggi:", pendapatan_per_kategori.first()["kategori"])

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+

Kategori dengan total_pendapatan tertinggi: Rumah Tangga


D2. Kota dengan jumlah transaksi tier "Besar" terbanyak

In [14]:
tier_besar_per_kota = df.filter(col("tier_transaksi") == "Besar") .groupBy("kota") .agg(count("order_id").alias("jumlah_transaksi_besar")) .orderBy(col("jumlah_transaksi_besar").desc())

tier_besar_per_kota.show()
print("Kota dengan transaksi tier Besar terbanyak:", tier_besar_per_kota.first()["kota"])

+----------+----------------------+
|      kota|jumlah_transaksi_besar|
+----------+----------------------+
|      Solo|                    92|
|  Magelang|                    78|
|   Kebumen|                    78|
|Yogyakarta|                    75|
| Purworejo|                    66|
|  Semarang|                    65|
+----------+----------------------+

Kota dengan transaksi tier Besar terbanyak: Solo


D3. Rata-rata rating per metode_pembayaran

In [15]:
rating_per_metode = df.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).orderBy(col("rata_rata_rating").desc())

rating_per_metode.show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.168127490039839|
|    Transfer Bank|4.160079051383398|
|         E-Wallet|4.138599999999997|
|     Kartu Kredit|4.118902439024387|
+-----------------+-----------------+



E. Menyimpan Hasil ke HDFS 

Menyimpan DataFrame hasil bagian C (dengan kolom total_pendapatan dan tier_transaksi) ke HDFS dalam format CSV baru.

In [10]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan"

df.write.mode("overwrite").option("header", True).csv(output_path)
print("Berhasil disimpan ke:", output_path)

Berhasil disimpan ke: hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan


In [11]:
# Verifikasi hasil penyimpanan di HDFS
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan

Found 2 items
-rw-r--r--   3 xcell supergroup          0 2026-09-10 18:23 /user/mahasiswa/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 xcell supergroup      97500 2026-09-10 18:23 /user/mahasiswa/tugas4/hasil_olahan/part-00000-58877233-e470-4966-a8b7-8ae21ec52b85-c000.csv


Mengapa hasilnya berupa beberapa berkas partisi (part-00000..., dst), bukan satu berkas tunggal?

Spark memproses data secara terdistribusi — DataFrame dibagi ke beberapa partisi yang ditulis secara paralel oleh masing-masing task/executor, sehingga muncul banyak berkas part-XXXXX bukan satu file. Ini justru menunjukkan kekuatan Spark: penulisan tidak perlu menunggu semua data terkumpul di satu proses seperti pandas to_csv(), sehingga jauh lebih efisien untuk data berskala besar

In [3]:
spark.stop()
print("SparkSession Ditutup.")

SparkSession Ditutup.
